# Lesson 3: Cryptography & Secrets in Code

**Goal:** Understand how to protect data using Python. We will learn about **functions** to organize code, **hashing** to secure passwords, and **encryption** to keep secrets safe.

## Today's Menu:
1. **Homework Review:** Dictionary Attacks.
2. **Python Skills:** Functions & Modules.
3. **Hashing:** One-way security (MD5, SHA-256).
4. **Encryption:** Two-way security (Symmetric Encryption).
5. **Project:** Building a Password Manager.
6. **Real-world Breaches:** Why salting matters.

## 1. Python Power-Up: Functions

So far, we've written scripts that run from top to bottom. But what if we want to reuse a piece of code multiple times? Copy-pasting is bad practice. instead, we use **Functions**.

A function is a block of reusable code. You define it once, and call it whenever you need it.

### Anatomy of a Function
```python
def function_name(parameter):
    # Code to run
    result = parameter * 2
    return result
```
- `def`: tells Python we are defining a function.
- `parameter`: input data the function needs.
- `return`: output data the function gives back.

In [ ]:
# A simple function to check password length
def is_password_long_enough(password):
    if len(password) >= 8:
        return True
    else:
        return False

# Using the function
my_pass = "secret123"
check = is_password_long_enough(my_pass)
print(f"Is '{my_pass}' long enough? {check}")

short_pass = "1234"
print(f"Is '{short_pass}' long enough? {is_password_long_enough(short_pass)}")

## 2. Hashing: One-Way Streets

**Hashing** takes an input (like a password) and turns it into a fixed-length string of characters (the hash).
Crucially, **you cannot turn the hash back into the password.** It is a one-way process.

This is perfect for storing passwords. We don't store the password itself; we store the hash. When a user logs in, we hash their input and compare it to the stored hash.

We will use the `hashlib` library.
*   **MD5:** Older, broken, fast. (Do not use for security!)
*   **SHA-256:** Standard, secure.


In [ ]:
import hashlib

password = "supersecretpassword"

# MD5 Hashing (Broken, purely for example)
# Note: we must encode the string to bytes (b"string") before hashing
md5_hash = hashlib.md5(password.encode()).hexdigest()
print(f"MD5: {md5_hash}")

# SHA-256 Hashing (Secure)
sha256_hash = hashlib.sha256(password.encode()).hexdigest()
print(f"SHA-256: {sha256_hash}")

# Even a tiny change changes the whole hash
wrong_password = "supersecretpasswore" # changed 'd' to 'e'
new_hash = hashlib.sha256(wrong_password.encode()).hexdigest()
print(f"SHA-256 (modified): {new_hash}")


### The Problem with Unsalted Hashes: Rainbow Tables

If two users have the same password, they will have the same hash. Hackers use "Rainbow Tables" (pre-computed lists of hashes for common passwords) to reverse-engineer widely used passwords instantly.

**Solution: Salting.**
A **salt** is random data added to the password before hashing. This makes the hash unique even if the password is common.


In [ ]:
import os

# Generate a random salt (16 bytes)
salt = os.urandom(16)
print(f"Salt (bytes): {salt}")

# Combine salt and password
salted_password = salt + password.encode()

# Hash the salted password
secure_hash = hashlib.sha256(salted_password).hexdigest()
print(f"Salted SHA-256: {secure_hash}")

## 3. Encryption: Two-Way Security

Unlike Hashing, **Encryption** is reversible. You can lock data (encrypt) and unlock it (decrypt) if you have the **Key**.

We will use the `cryptography` library. Specifically, **Fernet** (symmetric encryption).
"Symmetric" means the *same key* is used to lock and unlock.


In [ ]:
from cryptography.fernet import Fernet

# 1. Generate a Key (Like making a physical key)
key = Fernet.generate_key()
print(f"Key: {key}")

# 2. Create the Lock (Cipher suite)
cipher = Fernet(key)

# 3. Encrypt a message
message = b"This is a secret message!"
encrypted_message = cipher.encrypt(message)
print(f"Encrypted: {encrypted_message}")

# 4. Decrypt the message
decrypted_message = cipher.decrypt(encrypted_message)
print(f"Decrypted: {decrypted_message}")


## 4. Project: The Simple Password Vault

We will build a tool that:
1.  Takes a service name (e.g., "Netflix") and a password.
2.  Encrypts the password.
3.  Saves it to a file.
4.  Can read it back later.

We will use **Functions** to organize this.


In [ ]:
# Setup
key = Fernet.generate_key()
cipher = Fernet(key)
VAULT_FILE = "my_passwords.txt"

def save_password(service, username, password):
    encrypted_pass = cipher.encrypt(password.encode()).decode() # decode bytes to string for writing
    entry = f"{service},{username},{encrypted_pass}\n"
    
    with open(VAULT_FILE, "a") as f:
        f.write(entry)
    print(f"Saved password for {service}!")

def get_password(service_name):
    with open(VAULT_FILE, "r") as f:
        for line in f:
            parts = line.strip().split(",")
            service = parts[0]
            username = parts[1]
            encrypted_pass = parts[2]
            
            if service == service_name:
                decrypted_pass = cipher.decrypt(encrypted_pass.encode()).decode()
                print(f"Found {service}!")
                print(f"User: {username}")
                print(f"Password: {decrypted_pass}")
                return

    print("Service not found.")

# Let's test it!
save_password("Instagram", "cool_user", "password123")
save_password("Bank", "rich_user", "securePass!99")

print("\n--- Retrieving ---")
get_password("Instagram")


## 5. Real-World Failures: Why this matters

-   **LinkedIn (2012):** stored 6.5 million passwords as **unsalted SHA-1 hashes**. Hackers downloaded the database and cracked millions of them quickly using Rainbow Tables.
-   **RockYou (2009):** Stored 32 million passwords in **plaintext** (no encryption, no hashing). Anyone who saw the database saw every password immediately.

**Lesson:**
-   **Passwords:** Always Hash + Salt.
-   **Sensitive Data:** Encrypt.
-   **Never** store plaintext passwords.
